Definição do repositório

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/drive


In [ ]:
base = "drive/My Drive/Colab Notebooks"
repository = "refactoring-toy-example"

In [ ]:
import json
import csv

class File:

  def __init__(self, path):
    self.path = path

  def read(self):
    with open(self.path, "r") as file:
      if self.path.endswith(".json"):
        return json.load(file)
      elif self.path.endswith(".csv"):
        result = []
        reader = csv.DictReader(file)
        for row in reader:
          result.append(row)
        return result
        
    return None

  def write(self, data):
    with open(self.path, "w") as file:
      if self.path.endswith(".json"):
        json.dump(data, file, indent = 2, sort_keys=True)
        print(f"> write {self.path}")

import os

class Folder:

  def __init__(self, base):
    self.base = base

  def all_files(self):
    result = []
    self.__all_files_recursive(result, self.base)
    return result

  def __all_files_recursive(self, result, folder):
    for file in os.listdir(folder):
      if os.path.isdir(f"{folder}/{file}"):
        self.__all_files_recursive(result, f"{folder}/{file}")
      else: 
        filepath = os.path.join(folder, file)
        result.append(filepath)
        

  def all_files_by_ext(self, ext=".json"):
    result = []
    for file in self.all_files():
      if file.endswith(ext):
        result.append(file)
    return result

  def all_files_with_pattern(self, pattern):
    result = []
    for file in self.all_files():
      if pattern in file:
        result.append(file)
    return result
  
  def file(self, path):
    return File(path)

  def create(self):
    if not os.path.exists(self.base):
      os.makedirs(self.base)

  

Listagem de grupos de modificações

In [ ]:
file = File(f"{base}/metadata/{repository}/clusters-manutenabilidade.json")
lista_grupos = list(file.read().keys())
#print(lista_grupos)

In [ ]:
#ler arquivo zip
import zipfile
import json

#multipattern_zip = zipfile.ZipFile(f"{base}/metadata/{repository}/multipattern.zip", "r")

def read_file_from_zip(zipfile, file):
  data = zipfile.read(file)
  return json.loads(data.decode("utf-8"))


In [ ]:
clusters_zip = zipfile.ZipFile(f"{base}/metadata/{repository}/clusters.zip", "r")

In [ ]:

grupos = {}
for item in lista_grupos:
    file = read_file_from_zip(clusters_zip, f"clusters/{item}.json")
    #file = File(f"{base}/metadata/{repository}/clusters/{item}.json").read()
    #if len(file["items"]) > 1:
    grupos[item] = file["items"]
#print(grupos)
grupos_chaves = list(grupos.keys())
#print(grupos_chaves)
clusters_zip.close()

Função de leitura de multipatterns

In [ ]:
file = File(f"{base}/metadata/{repository}/analise-multipattern.json").read()
multipatterns = {}

multipattern_zip = zipfile.ZipFile(f"{base}/metadata/{repository}/multipattern.zip", "r")

medidas = {}
for commit,values in file.items():
    for multi,medida in values.items():
        multipatterns[multi] = commit
        medidas[multi] = medida
                
#print(multipatterns)
#print(medidas)
  
def leitura_multipattern(code):
    commit = multipatterns[code]
    file = read_file_from_zip(multipattern_zip, f"multipattern/{commit}/{code}.json")
    #file = File(f"{base}/metadata/{repository}/multipattern/{commit}/{code}.json").read()
    return file
    

Organizar os dados de modificação de cada arquivo

In [ ]:
files = {}

def executar(commit, arquivo, multipattern_code, modificacao, impacto):
    valor = medidas[f"#{multipattern_code}"]
    key = f"{commit}-{arquivo}"
    if key in files:
        if modificacao not in files[key]["items"].keys():
            files[key]["items"][modificacao] = 0
            
        files[key]["items"][modificacao] = files[key]["items"][modificacao] + impacto
        
        if valor > files[key]["result"]:
            files[key]["result"] = valor
            
    else:
        files[key] = {}
        files[key]["result"] = valor
        files[key]["items"] = {}
        files[key]["items"][modificacao] = impacto

def aplicar(item, count, modificacao, multipattern_code):
    tipo = item["type"]
    commit = multi["commits"][count]
    '''
    if count > 0:
        prev_commit = multi["commits"][count-1]
    else:
        posicao = commits_geral.index(commit)
        prev_commit = commits_geral[posicao-1]
    '''
    newFile = None
    oldFile = None
    if "newFile" in item:
        newFile = item["newFile"]
    if "oldFile" in item:
        oldFile = item["oldFile"]
    
    #executar(commit, arquivo, code, modificacao, impacto)
    if tipo == "delete":
        executar(commit, oldFile, multipattern_code, modificacao, 1)
    elif tipo == "insert":
        executar(commit, newFile, multipattern_code, modificacao, 1)
    else:
        if oldFile == newFile: 
            executar(commit, oldFile, multipattern_code, modificacao, 1)
        else:
            executar(commit, oldFile, multipattern_code, modificacao, -1)
            executar(commit, newFile, multipattern_code, modificacao, 1)
        
    

for modificacao, ocorrencias in grupos.items():
    for ocorrencia in ocorrencias:
        multi = leitura_multipattern(f"#{ocorrencia}")
        count = 0
        for item in multi["items"]:
            aplicar(item, count, modificacao, ocorrencia)
            count = count + 1
            
                
print(f"Número de arquivos: {len(files)}")
multipattern_zip.close()


Número de arquivos: 154


Define função de criação de instância no modelo

In [ ]:
def criar_instancia(name, element):
    item = {}
    item["name"] = name
    for chave in grupos_chaves:
        item[chave] = 0
    for modificacao,impacto in element["items"].items():
        #item[modificacao] = impacto
        
        if impacto > 0:
            item[modificacao] = 1
        else:
            item[modificacao] = -1
            
    item["result"] = element["result"]
    
    return item

Cria arquivo do modelo

In [ ]:
import pandas as pd

result = []
for name, file in files.items():
    item = criar_instancia(name,file)
    result.append(item)
    
modelo = pd.DataFrame.from_records(data=result)
path = f"{base}/metadata/{repository}/modelo.csv"
modelo.to_csv(path, index=False)

In [ ]:
#data = pd.read_csv(path)
#data